In [1]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews   -p .

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [2]:
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import nltk
from collections import Counter
from nltk.tokenize import TreebankWordTokenizer

nltk.download('punkt')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


Using device: cuda


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# with zipfile.ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
#     zip_ref.extractall(".")  

In [4]:
df =pd.read_csv("IMDB Dataset.csv")

import string
import re
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'<br\s*/?>', ' ', text)  
    text = text.translate(str.maketrans(' ', ' ', string.punctuation))  # Remove all punctuation
    text = re.sub(r'\s+', ' ', text).strip() 
    return text

df["review"] = df["review"].apply(preprocess_text)
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

print(df["review"][0])


one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far away i would say the main appeal of the show is due to the fact that it goes where other shows wouldnt dare forget pretty pict

In [5]:
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]
test_df.head()

,review,sentiment,label
45000,what i enjoyed most in this film was the scene...,positive,1
45001,macarthur is a great movie with a great story ...,positive,1
45002,what can i say i ignored the reviews and went ...,negative,0
45003,a pretty transparent attempt to wring cash out...,negative,0
45004,even though the book wasnt strictly accurate t...,negative,0


In [6]:
tokenizer = TreebankWordTokenizer()
counter=Counter()
for text in train_df["review"]:
    counter.update(tokenizer.tokenize(text))



In [7]:
vocab = {word: i + 2 for i, (word, _) in enumerate(counter.most_common())}
vocab["<unk>"] = 0
vocab["<pad>"] = 1

print(list(vocab.items())[:20])

[('the', 2), ('and', 3), ('a', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('it', 9), ('i', 10), ('this', 11), ('that', 12), ('was', 13), ('as', 14), ('with', 15), ('for', 16), ('movie', 17), ('but', 18), ('film', 19), ('on', 20), ('not', 21)]


In [8]:
class IMDBDataset(Dataset):
    def __init__(self, df, vocab):
        self.texts = df["review"].tolist()
        self.labels = df["label"].tolist()
        self.vocab = vocab
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        tokens = tokenizer.tokenize(self.texts[idx])
        token_ids = [self.vocab.get(token, self.vocab["<unk>"]) for token in tokens]
        label = self.labels[idx]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)


In [9]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.tensor(labels, dtype=torch.float)  
    return texts.to(device), labels.to(device)


train_dataset = IMDBDataset(train_df, vocab)
test_dataset = IMDBDataset(test_df, vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)


In [10]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, rnn_type="rnn"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab["<pad>"])
        if rnn_type == "rnn":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif rnn_type == "gru":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)  # Output one value per sample
    
    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, _ = self.rnn(embedded)
        out = self.fc(rnn_out[:, -1, :])
        return torch.sigmoid(out).squeeze() 


In [11]:
def train_and_evaluate(rnn_type,hidden_dim,lr,epoches):
    model = RNNModel(len(vocab), embed_dim=128, hidden_dim=hidden_dim, output_dim=1, rnn_type=rnn_type).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epoches):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in test_loader:
            outputs = model(texts)
            predicted = (outputs > 0.5).float()  
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"{rnn_type.upper()} Accuracy: {accuracy:.4f}")
    return accuracy


In [12]:
rnn_acc = train_and_evaluate("rnn",256,0.01,50)

Epoch 1, Loss: 0.7321
Epoch 2, Loss: 0.7303


KeyboardInterrupt: 

In [ ]:
gru_acc = train_and_evaluate("gru",256,20)

In [ ]:
lstm_acc = train_and_evaluate("lstm",256,20)

In [ ]:
print(f"\nPerformance Comparison:\nRNN: {rnn_acc:.4f}\nGRU: {gru_acc:.4f}\nLSTM: {lstm_acc:.4f}")
